In [ ]:
from google.colab import files
uploaded = files.upload()

Saving watch_history.csv to watch_history (1).csv
Saving watchlist.csv to watchlist (1).csv
Saving subscribers.csv to subscribers (1).csv
Saving titles.csv to titles (1).csv
Saving ratings.csv to ratings (1).csv
Saving reviews.csv to reviews (1).csv


In [ ]:
import pandas as pd
subscribers = pd.read_csv("/content/subscribers.csv")
titles = pd.read_csv("/content/titles.csv")
watch_history = pd.read_csv("/content/watch_history.csv")
ratings = pd.read_csv("/content/ratings.csv")
reviews = pd.read_csv("/content/reviews.csv")
watchlist = pd.read_csv("/content/watchlist.csv")

In [ ]:
print("Subscribers:", subscribers.shape)
print("Titles:", titles.shape)
print("Watch History:", watch_history.shape)
print("Ratings:", ratings.shape)
print("Reviews:", reviews.shape)
print("Watchlist:", watchlist.shape)

Subscribers: (15000, 14)
Titles: (9000, 21)
Watch History: (650000, 10)
Ratings: (130000, 5)
Reviews: (110000, 7)
Watchlist: (65000, 5)


**Finding:**
All six StreamFlix datasets were successfully loaded and profiled. The datasets contain 979,000 records in total across 6 tables: 15,000 subscribers, 9,000 titles, 650,000 watch-history records, 130,000 ratings, 110,000 reviews, and 65,000 watchlist records. The initial data-type inspection showed that numeric, boolean, and categorical fields were correctly represented, while date fields were initially stored as object/string values and were subsequently converted to datetime format.


In [ ]:
subscribers.info()
titles.info()
watch_history.info()
ratings.info()
reviews.info()
watchlist.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   subscriber_id      15000 non-null  object 
 1   signup_date        15000 non-null  object 
 2   country            15000 non-null  object 
 3   region             15000 non-null  object 
 4   age                15000 non-null  int64  
 5   gender             15000 non-null  object 
 6   plan_type          15000 non-null  object 
 7   monthly_price_usd  15000 non-null  float64
 8   household_size     15000 non-null  int64  
 9   primary_device     15000 non-null  object 
 10  payment_method     15000 non-null  object 
 11  tenure_months      15000 non-null  int64  
 12  is_active          15000 non-null  bool   
 13  churn_date         3801 non-null   object 
dtypes: bool(1), float64(1), int64(3), object(9)
memory usage: 1.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90

In [ ]:
datasets = {
    "Subscribers": subscribers,
    "Titles": titles,
    "Watch History": watch_history,
    "Ratings": ratings,
    "Reviews": reviews,
    "Watchlist": watchlist
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


Subscribers
subscriber_id            0
signup_date              0
country                  0
region                   0
age                      0
gender                   0
plan_type                0
monthly_price_usd        0
household_size           0
primary_device           0
payment_method           0
tenure_months            0
is_active                0
churn_date           11199
dtype: int64

Titles
title_id                   0
title_name                 0
type                       0
primary_genre              0
country                    0
language                   0
release_year               0
date_added                 0
maturity_rating            0
seasons                    0
content_duration_min       0
is_original                0
license_type               0
director                   0
cast                       0
quality_score              0
popularity_score           0
license_cost_usd           0
license_expiry          1966
total_watch_hours          0
total_pl

**Finding:**
Missing values were identified in two columns. subscriber churn date contains 11,199 missing values (74.66%), while title license expiry contains 1,966 missing values (21.84%). All other columns across the six datasets contain no missing values. These missing values were retained because they require business-rule interpretation rather than automatic replacement.


In [ ]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Subscribers: 0 duplicate rows
Titles: 0 duplicate rows
Watch History: 0 duplicate rows
Ratings: 0 duplicate rows
Reviews: 0 duplicate rows
Watchlist: 0 duplicate rows


In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")

    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100

    report = pd.DataFrame({
        "Missing Values": missing,
        "Missing %": missing_pct.round(2)
    })

    print(report[report["Missing Values"] > 0])


Subscribers
            Missing Values  Missing %
churn_date           11199      74.66

Titles
                Missing Values  Missing %
license_expiry            1966      21.84

Watch History
Empty DataFrame
Columns: [Missing Values, Missing %]
Index: []

Ratings
Empty DataFrame
Columns: [Missing Values, Missing %]
Index: []

Reviews
Empty DataFrame
Columns: [Missing Values, Missing %]
Index: []

Watchlist
Empty DataFrame
Columns: [Missing Values, Missing %]
Index: []


**Finding:**
No exact duplicate rows were found in any of the six datasets. In addition, no duplicate subscriber id values were found in the subscribers table and no duplicate watch id values were found in the watch-history table. Therefore, no duplicate records were removed during Phase 1.


In [ ]:
print("Duplicate subscriber_id:",
      subscribers["subscriber_id"].duplicated().sum())
print("Duplicate watch_id:",
      watch_history["watch_id"].duplicated().sum())

Duplicate subscriber_id: 0
Duplicate watch_id: 0


In [ ]:
date_columns = {
    "subscribers": ["signup_date", "churn_date"],
    "titles": ["date_added", "license_expiry"],
    "watch_history": ["watch_date"],
    "ratings": ["rating_date"],
    "reviews": ["review_date"],
    "watchlist": ["added_date"]
}

date_columns

{'subscribers': ['signup_date', 'churn_date'],
 'titles': ['date_added', 'license_expiry'],
 'watch_history': ['watch_date'],
 'ratings': ['rating_date'],
 'reviews': ['review_date'],
 'watchlist': ['added_date']}

In [ ]:
subscribers["signup_date"] = pd.to_datetime(
    subscribers["signup_date"], errors="coerce"
)

subscribers["churn_date"] = pd.to_datetime(
    subscribers["churn_date"], errors="coerce"
)

titles["date_added"] = pd.to_datetime(
    titles["date_added"], errors="coerce"
)

titles["license_expiry"] = pd.to_datetime(
    titles["license_expiry"], errors="coerce"
)

watch_history["watch_date"] = pd.to_datetime(
    watch_history["watch_date"], errors="coerce"
)

ratings["rating_date"] = pd.to_datetime(
    ratings["rating_date"], errors="coerce"
)

reviews["review_date"] = pd.to_datetime(
    reviews["review_date"], errors="coerce"
)

watchlist["added_date"] = pd.to_datetime(
    watchlist["added_date"], errors="coerce"
)

**Finding:**
The required date columns were initially stored as object/string values. They were converted to datetime format for proper date-based analysis. The existing missing values in churn date and license expiry were preserved as NaT. No additional invalid date values were identified during conversion.


In [ ]:
print("Subscribers signup_date invalid:",
      subscribers["signup_date"].isna().sum())

print("Subscribers churn_date invalid:",
      subscribers["churn_date"].isna().sum())

print("Titles date_added invalid:",
      titles["date_added"].isna().sum())

print("Titles license_expiry invalid:",
      titles["license_expiry"].isna().sum())

print("Watch history watch_date invalid:",
      watch_history["watch_date"].isna().sum())

print("Ratings rating_date invalid:",
      ratings["rating_date"].isna().sum())

print("Reviews review_date invalid:",
      reviews["review_date"].isna().sum())

print("Watchlist added_date invalid:",
      watchlist["added_date"].isna().sum())

Subscribers signup_date invalid: 0
Subscribers churn_date invalid: 11199
Titles date_added invalid: 0
Titles license_expiry invalid: 1966
Watch history watch_date invalid: 0
Ratings rating_date invalid: 0
Reviews review_date invalid: 0
Watchlist added_date invalid: 0


In [ ]:
duration_issues = watch_history[
    watch_history["watch_duration_min"] > watch_history["content_duration_min"]
]

print("Sessions longer than content:",
      len(duration_issues))

Sessions longer than content: 0


In [ ]:
duration_issue_pct = (
    len(duration_issues) / len(watch_history)
) * 100

print("Percentage:",
      round(duration_issue_pct, 2), "%")

Percentage: 0.0 %


In [ ]:
duration_issues.head(10)

,watch_id,subscriber_id,title_id,watch_date,device,region,content_duration_min,watch_duration_min,completion_pct,completed


In [ ]:
invalid_subscribers = watch_history[
    ~watch_history["subscriber_id"].isin(
        subscribers["subscriber_id"]
    )
]

print("Invalid subscriber IDs:",
      len(invalid_subscribers))

Invalid subscriber IDs: 0


In [ ]:
invalid_titles = watch_history[
    ~watch_history["title_id"].isin(
        titles["title_id"]
    )
]

print("Invalid title IDs:",
      len(invalid_titles))

Invalid title IDs: 0


In [ ]:
churned = subscribers[
    subscribers["churn_date"].notna()
].copy()

print("Churned subscribers:", len(churned))

Churned subscribers: 3801


In [ ]:
invalid_churn_dates = churned[
    churned["churn_date"] <= churned["signup_date"]
]

print("Invalid churn dates:",
      len(invalid_churn_dates))

Invalid churn dates: 0


Finding: There are 3,801 subscribers with a recorded churn date. All churn dates occur after the corresponding signup dates, so no invalid chronological relationships were identified.

In [ ]:
active_with_churn = subscribers[
    (subscribers["is_active"] == True) &
    (subscribers["churn_date"].notna())
]

print("Active subscribers with churn date:",
      len(active_with_churn))
inactive_without_churn = subscribers[
    (subscribers["is_active"] == False) &
    (subscribers["churn_date"].isna())
]

print("Inactive subscribers without churn date:",
      len(inactive_without_churn))

Active subscribers with churn date: 0
Inactive subscribers without churn date: 0


Finding: All active subscribers have a blank churn_date, and all inactive subscribers have a recorded churn_date. No mismatches were identified.

In [ ]:
short_tenure_active = subscribers[
    (subscribers["is_active"] == True) &
    (subscribers["tenure_months"] < 3)
]

print("Short-tenure active subscribers:",
      len(short_tenure_active))
short_tenure_active["tenure_months"].value_counts().sort_index()

Short-tenure active subscribers: 486


,count
tenure_months,
1,321
2,165


Finding: 486 active subscribers have a tenure of less than 3 months. Of these, 321 have 1 month of tenure and 165 have 2 months. These records were considered valid and were retained.

In [ ]:
watch_history["calculated_completion_pct"] = (
    watch_history["watch_duration_min"] /
    watch_history["content_duration_min"]
) * 100
watch_history["completion_difference"] = (
    watch_history["completion_pct"] -
    watch_history["calculated_completion_pct"]
).abs()
completion_issues = watch_history[
    watch_history["completion_difference"] > 1
]

print("Completion percentage mismatches:",
      len(completion_issues))

print(
    "Mismatch percentage:",
    round(len(completion_issues) / len(watch_history) * 100, 2),
    "%"
)


Completion percentage mismatches: 0
Mismatch percentage: 0.0 %


Finding: The calculated completion percentage was compared with the recorded completion_pct for all 650,000 watch-history records. No records differed by more than 1 percentage point (0.00% mismatch), indicating that the completion percentage is consistent with the recorded watch and content durations.

In [ ]:
watch_history.drop(
    columns=["calculated_completion_pct", "completion_difference"],
    inplace=True
)

In [ ]:
reviews["sentiment"].value_counts()

,count
sentiment,
Positive,75007
Neutral,22759
Negative,12234


In [ ]:
print("Unique sentiment values:")
print(reviews["sentiment"].unique())

Unique sentiment values:
['Positive' 'Negative' 'Neutral']


Finding: The sentiment column contains only the expected categories: Positive, Neutral, and Negative. No invalid sentiment values were identified across all 110,000 reviews. The records were retained without modification.

In [ ]:
invalid_watchlist_subscribers = watchlist[
    ~watchlist["subscriber_id"].isin(
        subscribers["subscriber_id"]
    )
]

print(
    "Invalid watchlist subscriber IDs:",
    len(invalid_watchlist_subscribers)
)

Invalid watchlist subscriber IDs: 0


In [ ]:
invalid_watchlist_titles = watchlist[
    ~watchlist["title_id"].isin(
        titles["title_id"]
    )
]

print(
    "Invalid watchlist title IDs:",
    len(invalid_watchlist_titles)
)

Invalid watchlist title IDs: 0


Finding: All 65,000 watchlist records were checked for referential integrity. Every subscriber_id exists in the subscribers table and every title_id exists in the titles table. No invalid references were identified.

## Data Quality Report

The StreamFlix project contains six datasets with a total of 979,000 records: 15,000 subscribers, 9,000 titles, 650,000 watch-history records, 130,000 ratings, 110,000 reviews, and 65,000 watchlist records.

### Data Completeness

Missing values were identified in two columns. `churn_date` in the subscribers table contains 11,199 missing values (74.66%), while `license_expiry` in the titles table contains 1,966 missing values (21.84%). All other columns contain no missing values. These missing values were retained because they require business-rule interpretation.

### Duplicates and Data Types

No duplicate rows were found in any dataset. There were also no duplicate `subscriber_id` values in the subscribers table or duplicate `watch_id` values in the watch-history table. Required date fields were converted from object/string format to datetime format, with no additional invalid dates identified.

### Data Validation

No watch-history sessions had a `watch_duration_min` greater than the corresponding `content_duration_min`. Referential-integrity checks found no invalid subscriber or title references in the watch-history table or watchlist table.

### Business Rule Validation

All 3,801 churned subscribers had a `churn_date` after their `signup_date`. No active subscribers had a churn date, and no inactive subscribers were missing a churn date. A total of 486 active subscribers had less than three months of tenure; these records were considered valid and retained as instructed by the PRD.

The recorded `completion_pct` was consistent with the calculated value based on watch duration and content duration, with no mismatches exceeding one percentage point. The review sentiment field contained only the expected values: Positive, Neutral, and Negative.

### Conclusion

Overall, the datasets passed the required Phase 1 data-quality checks. No duplicate records, invalid key relationships, watch-duration outliers, invalid churn-date relationships, completion-percentage mismatches, or invalid sentiment categories were identified. The identified missing values were documented and retained rather than removed or artificially replaced.


In [ ]:
import os

os.makedirs("/content/streamflix_cleaned", exist_ok=True)

datasets = {
    "subscribers_cleaned.csv": subscribers,
    "titles_cleaned.csv": titles,
    "watch_history_cleaned.csv": watch_history,
    "ratings_cleaned.csv": ratings,
    "reviews_cleaned.csv": reviews,
    "watchlist_cleaned.csv": watchlist
}

for filename, df in datasets.items():
    df.to_csv(
        f"/content/streamflix_cleaned/{filename}",
        index=False
    )
    print(f"Saved: {filename} | Rows: {len(df):,}")

Saved: subscribers_cleaned.csv | Rows: 15,000
Saved: titles_cleaned.csv | Rows: 9,000
Saved: watch_history_cleaned.csv | Rows: 650,000
Saved: ratings_cleaned.csv | Rows: 130,000
Saved: reviews_cleaned.csv | Rows: 110,000
Saved: watchlist_cleaned.csv | Rows: 65,000


In [ ]:

import shutil
from google.colab import files

# Compress the six cleaned CSV files into one ZIP
shutil.make_archive(
    "/content/StreamFlix_Cleaned_Data",
    "zip",
    "/content/streamflix_cleaned"
)

# Download the ZIP to your laptop
files.download("/content/StreamFlix_Cleaned_Data.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>